In [0]:
# Databricks notebook source
# =====================================================
# 1️⃣ Define Connection Variables (EDIT THESE)
# =====================================================
'''A Secret Scope is a secure container in Databricks where you store sensitive information.
Variable	Meaning
sqlserver_host	Azure SQL Server address
sqlserver_port	SQL Server connection port
sqlserver_database	Database name
sqlserver_user	Database username
sqlserver_password	Database password
secret_scope_name	Name of the secret container
secret_key_name	Name of the secret entry

Why do we define these variables?

Because we need these details to establish a JDBC connection between Databricks and Azure SQL Server.
JDBC is used to connect Databricks with Azure SQL Server so that you can read data from Azure SQL and process it using PySpark.

'''

sqlserver_host = "muskanserver.database.windows.net"
sqlserver_port = "1433"
sqlserver_database = "demodb"
sqlserver_user = "muskan"
sqlserver_password = "Admin@17"

# Secret scope name (will be created if not exists)
secret_scope_name = "banking-scope"

# Secret key name (single secret containing full JSON)
secret_key_name = "sqlserver-connection-json"
# secret_key_name = "sqlserver-connection-json-dummy"

# COMMAND ----------


In [0]:
'''Why are we converting connection details into JSON?

Because instead of storing six different secrets, we can store all SQL Server connection information inside one secret.'''
import json

connection_config = {
    "host": sqlserver_host,
    "port": sqlserver_port,
    "database": sqlserver_database,
    "user": sqlserver_user,
    "password": sqlserver_password,
    "driver": "com.microsoft.sqlserver.jdbc.SQLServerDriver"
}

connection_json = json.dumps(connection_config)
'''What does json.dumps() do?
It converts a Python dictionary into a JSON string.
Before: Python dictionary
{
    "host": "server",
    "port": "1433"
}
After: JSON string
"{\"host\":\"server\",\"port\":\"1433\"}"
'''

print("Generated JSON Configuration:")
print(connection_json)

In [0]:
'''Why do we need these?

Because you are going to create a Secret Scope using the Databricks REST API.

To call a REST API, you generally need:
1. API URL – Where should the request go?
2. API Token – Who is making the request?  
a REST API allows two different applications to communicate with each other over the internet using HTTP requests.
'''
ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()

api_url  = ctx.apiUrl().getOrElse(None)     # e.g. https://adb-...azuredatabricks.net
api_token = ctx.apiToken().getOrElse(None)  # personal access token for this session

print(api_url)
print(api_token)  # handle securely, do not log in real code



In [0]:
import requests
import json

# ----------------------------------------
# Configuration
# ----------------------------------------
DATABRICKS_INSTANCE = api_url  # Replace with your workspace URL
DATABRICKS_TOKEN = api_token  # Replace with your PAT

scope_name = secret_scope_name  # Scope to be created
backend_type = "DATABRICKS"     # Use "AZURE_KEYVAULT" if integrating with Key Vault
url = f"{DATABRICKS_INSTANCE}/api/2.0/secrets/scopes/create"
'''You are calling the Databricks REST API to create a secret scope.

Notice this endpoint:'''

In [0]:


headers = {
    "Authorization": f"Bearer {DATABRICKS_TOKEN}",
    "Content-Type": "application/json"
}
'''Authorization: Provides authentication.
Bearer token: Proves that you have permission to call the API.
Content-Type: Tells the API that we are sending JSON.'''
payload = {
    "scope": scope_name
}


# Send request
# ----------------------------------------
response = requests.post(url, headers=headers, data=json.dumps(payload))

'''What does requests.post() do?

It sends a POST request to Databricks.
GET = Ask the server to give you data.
POST = Send data to the server.
In simple language:
"Databricks, please create my secret scope."
'''
# ----------------------------------------
# Handle response
# ----------------------------------------
if response.status_code == 200:
    print(f"Secret scope '{scope_name}' created successfully.")
else:
    print("Failed to create secret scope.")
    print("Status Code:", response.status_code)
    print("Response:", response.text)


# COMMAND ----------
'''Store SQL Server connection JSON in Secret Scope
scope	banking-scope
secret_key	sqlserver-connection-json
secret_value	Complete SQL Server JSON configuration'''
import requests
import json

scope = secret_scope_name          # Already existing scope
secret_key = secret_key_name       # Name of the secret entry
secret_value = connection_json # Value to store securely

# -------------------------------------------------
# API Endpoint
# -------------------------------------------------
url = f"{DATABRICKS_INSTANCE}/api/2.0/secrets/put"

headers = {
    "Authorization": f"Bearer {DATABRICKS_TOKEN}",
    "Content-Type": "application/json"
}

payload = {
    "scope": scope,
    "key": secret_key,
    "string_value": secret_value
}
'''scope: Where should the secret be stored?
key: What is the secret's name?
string_value: What actual information should be stored?'''
# -------------------------------------------------
# Send Request
# -------------------------------------------------
response = requests.post(url, headers=headers, data=json.dumps(payload))

# -------------------------------------------------
# Output
# -------------------------------------------------
if response.status_code == 200:
    print(f"Secret '{secret_key}' created successfully in scope '{scope}'.")
else:
    print("Failed to create secret.")
    print("Status:", response.status_code)
    print("Response:", response.text)


# COMMAND ----------

# =====================================================
# 5️⃣ Verify Secret Retrieval
# =====================================================

try:
    retrieved_json = dbutils.secrets.get(
        scope=secret_scope_name,
        key=secret_key_name
    )
    '''You are checking whether the secret can actually be retrieved.
    It retrieves a secret from Databricks.

It requires two things:
- Scope name.
- Secret key name.'''
    
    print("Secret retrieved successfully.")
    
    parsed = json.loads(retrieved_json)
    print("Parsed JSON:")
    print(parsed)
    #JSON string → Python dictionary
except Exception as e:
    print("Secret verification failed:")
    print(str(e))


# COMMAND ----------

import requests
import json

scope = secret_scope_name          # Already existing scope
secret_key = 'gmail_api_key'       # Name of the secret entry
secret_value = 'glhoxdugdcepffhg' # Value to store securely

# -------------------------------------------------
# API Endpoint
# -------------------------------------------------
url = f"{DATABRICKS_INSTANCE}/api/2.0/secrets/put"

headers = {
    "Authorization": f"Bearer {DATABRICKS_TOKEN}",
    "Content-Type": "application/json"
}

payload = {
    "scope": scope,
    "key": secret_key,
    "string_value": secret_value
}

# -------------------------------------------------
# Send Request
# -------------------------------------------------
response = requests.post(url, headers=headers, data=json.dumps(payload))

# -------------------------------------------------
# Output
# -------------------------------------------------
if response.status_code == 200:
    print(f"Secret '{secret_key}' created successfully in scope '{scope}'.")
else:
    print("Failed to create secret.")
    print("Status:", response.status_code)
    print("Response:", response.text)
payload = {
    "scope": scope_name
}
